In [6]:
!pip install control
!pip install matplotlib

In [2]:
import control as clt
import numpy as np
import matplotlib.pyplot as plt

In [3]:
# Define pendulum constants
m = 0.2 # mass of pendulum
M = 0.5 # mass of cart
l = 0.3 # length of pendulum 
I = m*pow((2*l),2)/12 # Moment of inertia (COM) [m*(2l)ˆ2]/12
J = I + m*l*l # 0.006 + 0.2*0.3*0.3 Icom + mlˆ2 (Parallel axis theorm)
g = 9.8
print(I, J)

0.005999999999999999 0.023999999999999997


In [106]:
# dx/dt = Ax + Bu
"""
    (M+m)x'' + mL*Cos(phi).phi'' = u + ML*Sin(phi).(phi')ˆ2        
    mL*Cos(phi)x'' + (I+mlˆ2)*phi'' = mgLSin(phi)

    We calculate [x'' phi'']
"""
D = (M+m)*J - (m*m*l*l)
A = np.array([
    [0, 1, 0, 0],
    [0, 0, (m*m*g*l*l)/D, 0],
    [0, 0, 0, 1],
    [0, 0, (M+m)*m*g*l/D, 0]
])

B = np.array([
    [0],
    [(J)/D],
    [0],
    [m*l/D]
])
C = np.eye(4)
D = np.zeros((4,1))
Q = np.diag([1, 1, 1, 1])
R = np.array([[1]])
# print(A)
# print(B)
sys_c = clt.ss(A, B, C, D)
sys_d_150 = clt.c2d(sys_c, 0.250, method='zoh')
sys_d_5 = clt.c2d(sys_c, 0.005, method='zoh')

A_d_150 = sys_d_150.A
B_d_150 = sys_d_150.B

A_d_5 = sys_d_5.A
B_d_5 = sys_d_5.B

K_150,_,_ = clt.dlqr(A_d_150, B_d_150, Q, R)
K_150
A_cl_150 = A_d_150 - B_d_150*K_150
print(f"Check stability of loop for K: {K_150}")
if max(abs(np.linalg.eigvals(A_cl_150))) < 1:
    print("Stable")
else:
    print("Unstable")
for q in range(1, 1001):
    Q = np.diag([1, 1, q, 1])
    K_5, _, _ = clt.dlqr(A_d_5, B_d_5, Q, R)
    A_cl_150 = A_d_150 - B_d_150*K_5
    A_cl_5 = A_d_5 - B_d_5*K_5
    print(f"Check stability of loop for K: {K_5}")

    if max(abs(np.linalg.eigvals(A_cl_5))) > 1:
        continue
        
    if max(abs(np.linalg.eigvals(A_cl_150))) < 1:
        print("Stable")
    else:
        print("Unstable")
        print(Q)
        break

Check stability of loop for K: [[-0.17135705 -0.36722009  9.87643873  1.80561455]]
Stable
Check stability of loop for K: [[-0.96480715 -1.87180077 19.92649512  3.83957234]]
Unstable
[[1 0 0 0]
 [0 1 0 0]
 [0 0 1 0]
 [0 0 0 1]]


In [79]:
def dlqr(T, q):
  sys_c = clt.ss(A, B, C, D)
  sys_d = clt.c2d(sys_c, T, method='zoh')
  A_d = sys_d.A
  B_d = sys_d.B
  Q = np.diag([1, 1, q, 1])
  K, S, E = clt.dlqr(A_d, B_d, Q, R)

  A_cl = A_d - B_d*K
  if max(abs(np.linalg.eigvals(A_cl))) < 1:
        print(f"Stable gains:  ")
        gain = ', '.join([str(k) for k in K])
        print(gain)
  else:
        print("Unstable gain")
  return K, A_d, B_d

In [107]:
K_5, A_d_5, B_d_5 = dlqr(T=0.005, q=100)
K_150, A_d_150, B_d_150 = dlqr(T=0.150, q=100)
A_cl_150 = A_d_150 - B_d_150*K_5
print(max(abs(np.linalg.eigvals(A_cl_150))))


Stable gains:  
[-0.96183101 -2.06165895 23.80067011  4.17745663]
Stable gains:  
[-0.31661315 -0.71078538 13.12746744  2.35946295]
1.270125478875121
